In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.silver.order_items")
orders = spark.table("workspace.silver.orders")
products = spark.table("workspace.silver.products")

product_sales = (
    items
    .join(
        orders.select("order_id", "order_status"),
        "order_id",
        "inner"
    )
    .join(
        products.select("product_id", "product_name", "category"),
        "product_id",
        "left"
    )
    .filter(F.col("order_status").isin("COMPLETED", "SHIPPED"))
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("line_total").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

(
    product_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.product_sales")
)



In [0]:
display(product_sales)